# முகில்நோக்கு (MugizhNokku) - Colab Runner
Run this notebook in Google Colab with a GPU runtime to train the Digital Twin Predictor.

In [ ]:
# 1. Clone the GitHub repository
!rm -rf MugizhNokku
!git clone https://github.com/phk518/MugizhNooku.git MugizhNokku
%cd MugizhNokku
!pip install -r requirements.txt

In [ ]:
# 2. Mount Google Drive to save and load models and data
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/ISRO_Hackathon_PS5')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Project ready at: {PROJECT_ROOT}')

In [ ]:
# 3. Example: Launch the Streamlit dashboard via localtunnel
!npm install -g localtunnel
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

In [ ]:
from dataset import ClimateDataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from model import DigitalTwinPredictor

# 1. Initialize Dataset & DataLoader
# (Assuming 'ds_merged' is your preprocessed xarray combining Rainfall & Temp)
seq_length = 7
climate_data = ClimateDataset(ds_merged, sequence_length=seq_length, target_vars=['rainfall'])

# Use a small batch size to prevent GPU memory overload
train_loader = DataLoader(climate_data, batch_size=4, shuffle=True)

# 2. Dynamic Model Instantiation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")
print(f"Detected Channels: {climate_data.num_channels}")

# The model dynamically adjusts its input_channels based on the dataset
model = DigitalTwinPredictor(
    input_channels=climate_data.num_channels, 
    hidden_channels=32, 
    out_channels=len(climate_data.target_vars)
).to(device)

# 3. Optimizer & Loss Function
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Proof-of-Concept Training Loop (Single Batch Validation)
print("Initiating tensor forward pass...")
for batch_idx, (X, y) in enumerate(train_loader):
    # Move 5D Tensors to GPU
    X, y = X.to(device), y.to(device) 
    
    optimizer.zero_grad()
    
    # Forward Pass: X shape is (Batch, Seq_Len, Channels, Height, Width)
    predictions = model(X) 
    
    # Compute Loss against T+1 target
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()
    
    print(f"Success! Batch {batch_idx} processed. Current Loss: {loss.item():.4f}")
    print(f"Input Shape: {X.shape}")
    print(f"Output Shape: {predictions.shape}")
    break # Halting after 1 batch for PoC verification
